In [2]:
!pip install -q spacy pandas nltk networkx matplotlib
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

folder_path = "/content/drive/MyDrive/RST"

if os.path.exists(folder_path):
    print("Files inside the RST folder:\n")
    for file in os.listdir(folder_path):
        print(file)
else:
    print("Folder not found:", folder_path)

Files inside the RST folder:

bbc article.txt
AI generated.txt


In [5]:
import os


bbc_file = "/content/drive/MyDrive/RST/bbc article.txt"


if os.path.exists(bbc_file):
    print("BBC article found.\n")


    with open(bbc_file, "r", encoding="utf-8") as file:
        bbc_text = file.read()


    print(bbc_text[:2000])

else:
    print("File not found:", bbc_file)

BBC article found.

Navin Singh Khadka
Environment correspondent
Camels are famously known as the ships of the desert, but even these hardy animals are not being spared from the extremes of rising temperatures in Africa.
The Afar region of north-eastern Ethiopia is one of the hottest and driest places on Earth, where nomads often lead camel caravans carrying salt through the desert.
"I can see my camels suffering from extreme heat," Ali Umer, a camel herder from Semera in Afar, told the BBC.

"They get blisters on their feet, their eyes become watery and the hot sand burns their skin and eats up their hair when they are sitting," said the 40-year-old who owns around 500 camels.
"Within the past one month I have lost eight camel calves
because of the extreme heat, it has become much more intense and difficult.
"Even the wind, which used to bring cool air, now carries heat," he said, adding he had noticed these drastic changes in the temperature in recent years.
He is not alone in his ob

In [6]:

!pip install -q spacy
!python -m spacy download en_core_web_sm

import spacy
import pandas as pd


nlp = spacy.load("en_core_web_sm")


doc = nlp(bbc_text)


sentences = [
    sent.text.strip()
    for sent in doc.sents
    if sent.text.strip()
]


bbc_df = pd.DataFrame({
    "Unit_ID": range(1, len(sentences) + 1),
    "Text": sentences
})

print(f"Total possible RST units: {len(bbc_df)}")

bbc_df

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Total possible RST units: 41


,Unit_ID,Text
0,1,Navin Singh Khadka\nEnvironment correspondent\...
1,2,The Afar region of north-eastern Ethiopia is o...
2,3,"""I can see my camels suffering from extreme he..."
3,4,"""They get blisters on their feet, their eyes b..."
4,5,"""Within the past one month I have lost eight c..."
5,6,"""Even the wind, which used to bring cool air, ..."
6,7,He is not alone in his observations: scientist...
7,8,The higher temperatures also indirectly make c...
8,9,North Africa and the Horn of Africa host aroun...
9,10,"Over the years, farmers and cattle herders in ..."


In [7]:
import re
import spacy
import pandas as pd


clean_text = bbc_text

clean_text = clean_text.replace("Navin Singh Khadka", "")
clean_text = clean_text.replace("Environment correspondent", "")


clean_text = re.sub(r'\n+', ' ', clean_text)
clean_text = re.sub(r'(\w)\.([A-Z])', r'\1. \2', clean_text)
clean_text = re.sub(r'(\d)%', r'\1%', clean_text)


doc = nlp(clean_text)

sentences = [
    sent.text.strip()
    for sent in doc.sents
    if sent.text.strip()
]


bbc_df = pd.DataFrame({
    "Unit_ID": range(1, len(sentences) + 1),
    "Text": sentences
})

print(f"Total units: {len(bbc_df)}")
bbc_df.head(30)

Total units: 42


,Unit_ID,Text
0,1,Camels are famously known as the ships of the ...
1,2,The Afar region of north-eastern Ethiopia is o...
2,3,"""I can see my camels suffering from extreme he..."
3,4,"""They get blisters on their feet, their eyes b..."
4,5,"""Within the past one month I have lost eight c..."
5,6,"""Even the wind, which used to bring cool air, ..."
6,7,He is not alone in his observations: scientist...
7,8,The higher temperatures also indirectly make c...
8,9,North Africa and the Horn of Africa host aroun...
9,10,"Over the years, farmers and cattle herders in ..."


In [8]:


bbc_rst = bbc_df.copy()


bbc_rst["Role"] = ""
bbc_rst["Relation"] = ""
bbc_rst["Confidence"] = 0.0


for i, row in bbc_rst.iterrows():
    text = row["Text"].lower()


    if any(word in text for word in [
        "study", "research", "scientists", "veterinarians",
        "experts", "analysis", "report", "published"
    ]):
        bbc_rst.at[i, "Role"] = "Satellite"
        bbc_rst.at[i, "Relation"] = "Evidence"
        bbc_rst.at[i, "Confidence"] = 0.80


    elif any(phrase in text for phrase in [
        "because of", "as a result", "cause behind",
        "due to", "causing"
    ]):
        bbc_rst.at[i, "Role"] = "Satellite"
        bbc_rst.at[i, "Relation"] = "Cause"
        bbc_rst.at[i, "Confidence"] = 0.75


    elif any(word in text for word in [
        "but", "however", "although", "while"
    ]):
        bbc_rst.at[i, "Role"] = "Nucleus"
        bbc_rst.at[i, "Relation"] = "Contrast"
        bbc_rst.at[i, "Confidence"] = 0.70


    elif any(word in text for word in [
        "region", "africa", "ethiopia", "somalia",
        "kenya", "north africa"
    ]):
        bbc_rst.at[i, "Role"] = "Satellite"
        bbc_rst.at[i, "Relation"] = "Background"
        bbc_rst.at[i, "Confidence"] = 0.65


    else:
        bbc_rst.at[i, "Role"] = "Satellite"
        bbc_rst.at[i, "Relation"] = "Elaboration"
        bbc_rst.at[i, "Confidence"] = 0.50


pd.set_option("display.max_colwidth", 100)
bbc_rst

,Unit_ID,Text,Role,Relation,Confidence
0,1,"Camels are famously known as the ships of the desert, but even these hardy animals are not being...",Nucleus,Contrast,0.70
1,2,"The Afar region of north-eastern Ethiopia is one of the hottest and driest places on Earth, wher...",Satellite,Background,0.65
2,3,"""I can see my camels suffering from extreme heat,"" Ali Umer, a camel herder from Semera in Afar,...",Satellite,Elaboration,0.50
3,4,"""They get blisters on their feet, their eyes become watery and the hot sand burns their skin and...",Satellite,Elaboration,0.50
4,5,"""Within the past one month I have lost eight camel calves because of the extreme heat, it has be...",Satellite,Cause,0.75
5,6,"""Even the wind, which used to bring cool air, now carries heat,"" he said, adding he had noticed ...",Satellite,Elaboration,0.50
6,7,"He is not alone in his observations: scientists, animal welfare workers and various studies all ...",Satellite,Evidence,0.80
7,8,"The higher temperatures also indirectly make camels suffer by causing severe loss of water, vege...",Satellite,Cause,0.75
8,9,North Africa and the Horn of Africa host around 80% of the world's largest population of dromeda...,Satellite,Background,0.65
9,10,"Over the years, farmers and cattle herders in these regions have switched to these animals becau...",Satellite,Cause,0.75


In [13]:

markdown_table = bbc_rst.to_markdown(index=False)


print(markdown_table)

|   Unit_ID | Text                                                                                                                                                                                                                                                                                                                                                                                                                                                  | Role      | Relation    |   Confidence |
|----------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------|:------------|-------------:|
|         

In [14]:

markdown_table = bbc_rst.to_markdown(index=False)


with open("bbc_rst_analysis.md", "w", encoding="utf-8") as f:
    f.write(markdown_table)


from google.colab import files
files.download("bbc_rst_analysis.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:

ai_path = "/content/drive/MyDrive/RST/AI generated.txt"

with open(ai_path, "r", encoding="utf-8") as f:
    ai_text = f.read()


ai_units = [u.strip() for u in ai_text.split("\n\n") if u.strip()]


ai_df = pd.DataFrame({
    "Unit_ID": range(1, len(ai_units) + 1),
    "Text": ai_units
})

ai_df

,Unit_ID,Text
0,1,Even camels can't cope: Africa's ships of the desert hit by rising temperatures
1,2,"For centuries, the camel has been the ultimate survivor of Africa's drylands — an animal so well..."
2,3,"The animal renowned as the ""ship of the desert"" evolved to withstand extreme temperatures, water..."
3,4,## A herder's changing world
4,5,"Hassan Abdi Nur has kept camels in Kenya's Marsabit County for most of his life, following a rou..."
5,6,"""My father used to know exactly when the rains would come and where the grass would be green,"" N..."
6,7,"Nur said he lost four camels during the drought that struck the region between 2020 and 2023, on..."
7,8,"""We always said the camel does not die of thirst like other animals,"" he said. ""But I have seen ..."
8,9,Herders in neighbouring Somalia and Ethiopia's Somali region describe similar experiences. In pa...
9,10,## What the evidence shows


In [17]:


import re
import pandas as pd


ai_clean = ai_df[~ai_df["Text"].str.match(r"^##\s*", na=False)].copy()


ai_clean = ai_clean[ai_clean["Text"].str.strip() != ""]


ai_clean["Text"] = (
    ai_clean["Text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


ai_clean["Unit_ID"] = range(1, len(ai_clean) + 1)


ai_clean = ai_clean[["Unit_ID", "Text"]]


ai_clean

,Unit_ID,Text
0,1,Even camels can't cope: Africa's ships of the desert hit by rising temperatures
1,2,"For centuries, the camel has been the ultimate survivor of Africa's drylands — an animal so well..."
2,3,"The animal renowned as the ""ship of the desert"" evolved to withstand extreme temperatures, water..."
4,4,"Hassan Abdi Nur has kept camels in Kenya's Marsabit County for most of his life, following a rou..."
5,5,"""My father used to know exactly when the rains would come and where the grass would be green,"" N..."
6,6,"Nur said he lost four camels during the drought that struck the region between 2020 and 2023, on..."
7,7,"""We always said the camel does not die of thirst like other animals,"" he said. ""But I have seen ..."
8,8,Herders in neighbouring Somalia and Ethiopia's Somali region describe similar experiences. In pa...
10,9,Herders' testimony is increasingly backed by scientific data. Meteorological records for the Hor...
11,10,"Regional climate monitoring bodies, including IGAD's Climate Prediction and Applications Centre,..."


In [18]:


ai_rst = ai_clean.copy()

ai_rst["Role"] = ""
ai_rst["Relation"] = ""
ai_rst["Confidence"] = 0.0

for i, row in ai_rst.iterrows():
    text = row["Text"].lower()


    if any(word in text for word in [
        "study", "research", "scientists", "veterinarians",
        "experts", "analysis", "report", "published",
        "data", "researcher", "according to"
    ]):
        ai_rst.at[i, "Role"] = "Satellite"
        ai_rst.at[i, "Relation"] = "Evidence"
        ai_rst.at[i, "Confidence"] = 0.80


    elif any(phrase in text for phrase in [
        "because of", "as a result", "cause behind",
        "due to", "causing", "leads to", "affected by",
        "resulting in"
    ]):
        ai_rst.at[i, "Role"] = "Satellite"
        ai_rst.at[i, "Relation"] = "Cause"
        ai_rst.at[i, "Confidence"] = 0.75


    elif any(word in text for word in [
        "but", "however", "although", "while", "yet"
    ]):
        ai_rst.at[i, "Role"] = "Nucleus"
        ai_rst.at[i, "Relation"] = "Contrast"
        ai_rst.at[i, "Confidence"] = 0.70


    elif any(word in text for word in [
        "region", "africa", "ethiopia", "somalia",
        "kenya", "north africa", "sahara", "sahel"
    ]):
        ai_rst.at[i, "Role"] = "Satellite"
        ai_rst.at[i, "Relation"] = "Background"
        ai_rst.at[i, "Confidence"] = 0.65


    else:
        ai_rst.at[i, "Role"] = "Satellite"
        ai_rst.at[i, "Relation"] = "Elaboration"
        ai_rst.at[i, "Confidence"] = 0.50


ai_rst

,Unit_ID,Text,Role,Relation,Confidence
0,1,Even camels can't cope: Africa's ships of the desert hit by rising temperatures,Satellite,Background,0.65
1,2,"For centuries, the camel has been the ultimate survivor of Africa's drylands — an animal so well...",Satellite,Evidence,0.80
2,3,"The animal renowned as the ""ship of the desert"" evolved to withstand extreme temperatures, water...",Nucleus,Contrast,0.70
4,4,"Hassan Abdi Nur has kept camels in Kenya's Marsabit County for most of his life, following a rou...",Satellite,Background,0.65
5,5,"""My father used to know exactly when the rains would come and where the grass would be green,"" N...",Satellite,Elaboration,0.50
6,6,"Nur said he lost four camels during the drought that struck the region between 2020 and 2023, on...",Satellite,Background,0.65
7,7,"""We always said the camel does not die of thirst like other animals,"" he said. ""But I have seen ...",Nucleus,Contrast,0.70
8,8,Herders in neighbouring Somalia and Ethiopia's Somali region describe similar experiences. In pa...,Nucleus,Contrast,0.70
10,9,Herders' testimony is increasingly backed by scientific data. Meteorological records for the Hor...,Satellite,Evidence,0.80
11,10,"Regional climate monitoring bodies, including IGAD's Climate Prediction and Applications Centre,...",Satellite,Cause,0.75


In [21]:
# Convert the AI RST analysis table to Markdown and download it

from google.colab import files


ai_rst_md = ai_rst.to_markdown(index=False)


md_path = "/content/ai_rst_analysis.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write(ai_rst_md)


files.download(md_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>